# **RAG Pipelines: Data Ingestion to VectorDB Pipeline**

### Embedding And VectorStoreDB

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from pathlib import Path

In [6]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 390 - 73.Queue-LL.pdf
  ✓ Loaded 2 pages

Processing: 390 - 82.Selection-Sort.pdf
  ✓ Loaded 2 pages

Processing: Determinants 03 _ Class Notes - Prayas_2.0_Determinant_Rajeev Sir _ Lecture - 3 (1) 22-10-2021_compressed.pdf
  ✓ Loaded 30 pages

Processing: Work Energy Power 02 _ Class Notes - Work power energy lec-02 ;08-10-2021 (2) (2).pdf
  ✓ Loaded 36 pages

Total documents loaded: 70


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 10.14.2 (Build 18C54) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20190103112422Z00'00'", 'title': '120. Queue LL', 'moddate': "D:20190103112422Z00'00'", 'source': '..\\data\\pdf\\390 - 73.Queue-LL.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': '390 - 73.Queue-LL.pdf', 'file_type': 'pdf'}, page_content='Queue using Linked List\n#include <stdio.h>\n#include <stdlib.h>\nstruct Node\n{\n    int data;\n    struct Node *next;\n    \n}*front=NULL,*rear=NULL;\nvoid enqueue(int x)\n{\n    struct Node *t;\n    t=(struct Node*)malloc(sizeof(struct Node));\n    if(t==NULL)\n        printf("Queue is FUll\\n");\n    else\n    {\n        t->data=x;\n        t->next=NULL;\n        if(front==NULL)\n            front=rear=t;\n        else\n        {\n            rear->next=t;\n            rear=t;\n        }\n    }\n    \n}\nint dequeue()\n{\n    int x=-1;\n    struct Node* t;\n    \n    if(front==NULL)'),
 Document(meta

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import chromadb.config 
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize the embedding manager
        
        Args:
            model_name: Hugging Face model name for sentence transformer"""
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of documents

        Args:
            texts: List of document texts to embed

        Returns:
            numpy array of embeddings with shape (num_texts, embedding_dim)"""
        if self.model is None:
            raise ValueError("Embedding model is not loaded.")
        try:
            print(f"Generating embeddings for {len(texts)} documents...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print("Embeddings generated successfully.")
            return np.array(embeddings)
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise e


## Initializa the smbedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


f:\Placements\NLP\2_RAG\0_DataIngestion\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1816.94it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\HP\AppData\Local\Temp\ipykernel_14232\2968499620.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


## VectorStoreDB

In [ ]:
import os

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection to use
            persist_directory: Directory to persist the ChromaDB database
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # Create Persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings"}
            )

            print(f"Vector store initialized, Collection: {self.collection_name}" )
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain document objects (e.g., Document instances)
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        #Prepare data for chromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list = []

        for i,(doc,embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist()) # Convert numpy array to list for JSON serialization
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise 

vectorstore = VectorStore()
vectorstore

Vector store initialized, Collection: pdf_documents
Existing documents in collection: 0


In [6]:
# The variable `chunks` is not defined in this notebook.
# Define `chunks` first, for example by splitting documents into chunks,
# or use an existing variable such as `documents` or `vectorstore`.
print("Define 'chunks' before using it.")

Define 'chunks' before using it.
